# 🐦 Birdiary / wiediversistmeingarten.org: Starter Notebook

[wiediversistmeingarten.org](https://wiediversistmeingarten.org) runs a network of smart bird feeders.
Each station has a camera, a scale and a temperature/humidity sensor. When a bird lands on the perch, the station records
a short **video**, the bird's **weight** and the current **weather**. An AI model then guesses which species it was.

This notebook shows you how to:

1. Load **all stations** worldwide
2. Find the stations **in and around Heilbronn**
3. Explore a single station: its movements (bird visits), videos, daily statistics and sensor data
4. **Combine endpoints** to answer a real question: *"Where and when was a great tit (Parus major) seen in the last 30 days?"*

All the data is public, so you don't need an API key. Run the cells from top to bottom (`Shift + Enter`).

## The API at a glance

Base URL: `https://wiediversistmeingarten.org/api`

| Endpoint | What you get | Size |
|---|---|---|
| `GET /station` | All stations: id, name, location, type, last movement, last sensor reading | ~140 KB |
| `GET /station/{station_id}?movements=N&movementsOffset=K` | One station with daily species counts (`count`) and the **N newest movements** starting at position K (`measurements.movements`), plus paging info (`movementsMeta`) | small with `movements=…` |
| `GET /movement/{station_id}` | **All** movements of a station ever recorded (no paging) | can be several MB |
| `GET /statistics/{station_id}` | Summary statistics: total movements, average per day, min/max temperature & humidity, highlight videos per species | ~0.5 MB |
| `GET /environment/{station_id}` | **All** temperature/humidity readings (one every few minutes) | can be 15+ MB |

**What a movement looks like**

```json
{
  "mov_id": "4722d4ec-…",
  "station_id": "699b7f2c-…",
  "start_date": "2026-10-06 11:00:16.381854",   // when the bird arrived (station clock)
  "end_date":   "2026-10-06 11:00:19.224746",
  "weight": 13.1,                                // grams measured on the perch
  "detections": [                                // AI guesses, each with a confidence score 0…1
    {"latinName": "Cyanistes caeruleus", "germanName": "Blaumeise", "score": 0.61},
    {"latinName": "Parus major",         "germanName": "Kohlmeise", "score": 0.30}
  ],
  "validation": {…},                              // only present if a human checked the video
  "video": "https://wiediversistmeingarten.org/api/uploads/videos/….mp4",
  "audio": "https://wiediversistmeingarten.org/api/uploads/audios/….wav",
  "environment": {"temperature": 19.9, "humidity": 86.3, …}
}
```

## 0. Setup

In [ ]:
import math
import time
from datetime import datetime, timedelta

import pandas as pd
import requests

BASE_URL = "https://wiediversistmeingarten.org/api"

session = requests.Session()
session.headers.update({"User-Agent": "heilbronn-hackathon-starter-notebook"})


def get(path, **params):
    """GET an API path (e.g. "/station") and return the parsed JSON. Retries a few times on errors."""
    for attempt in range(3):
        try:
            response = session.get(f"{BASE_URL}{path}", params=params, timeout=60)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as error:
            if attempt == 2:
                raise
            print(f"Request failed ({error}), retrying …")
            time.sleep(2 * (attempt + 1))


pd.set_option("display.max_colwidth", 80)

## 1. All stations

`GET /station` returns a list of every station. We flatten it into a table (a pandas DataFrame).

In [ ]:
stations_raw = get("/station")
print(f"{len(stations_raw)} stations")

stations = pd.DataFrame(
    {
        "station_id": s["station_id"],
        "name": s.get("name"),
        "type": s.get("type"),
        "lat": (s.get("location") or {}).get("lat"),
        "lng": (s.get("location") or {}).get("lng"),
        "last_movement": (s.get("lastMovement") or {}).get("start_date"),
        "last_temperature": (s.get("lastEnvironment") or {}).get("temperature"),
        "last_humidity": (s.get("lastEnvironment") or {}).get("humidity"),
    }
    for s in stations_raw
)
stations["last_movement"] = pd.to_datetime(stations["last_movement"], format="ISO8601")

stations.sort_values("last_movement", ascending=False).head(10)

Not every station is active. Some have never recorded a bird (`last_movement` is empty), and some
have been offline for months:

In [ ]:
one_month_ago = datetime.now() - timedelta(days=30)

print("Stations with at least one movement:", stations["last_movement"].notna().sum())
print("Stations active in the last 30 days:", (stations["last_movement"] >= one_month_ago).sum())

### All stations on a map

[folium](https://python-visualization.github.io/folium/) draws interactive Leaflet maps. It comes preinstalled in Colab.
Green = active in the last 30 days, grey = inactive.

> **About the background map:** to keep this notebook free of API keys and usage limits, the maps here have **no background map**
> (`tiles=None`). You only see the station markers, which you can still pan, zoom and click. For your own project, look for a
> basemap that fits and check its usage policy, as many tile providers need an API key or a referrer.

In [ ]:
import folium

world_map = folium.Map(location=[51.0, 10.0], zoom_start=5, tiles=None)
for _, s in stations.dropna(subset=["lat", "lng"]).iterrows():
    active = pd.notna(s["last_movement"]) and s["last_movement"] >= one_month_ago
    folium.CircleMarker(
        location=[s["lat"], s["lng"]],
        radius=6,
        color="green" if active else "gray",
        fill=True,
        popup=f"<b>{s['name']}</b><br>last bird: {s['last_movement']}",
    ).add_to(world_map)
world_map

## 2. Stations in and around Heilbronn

The API has no "filter by city" option, so we filter by distance from the city centre ourselves.
Change `RADIUS_KM` to widen or narrow the search.

In [ ]:
HEILBRONN = (49.1427, 9.2109)  # lat, lng of the city centre
RADIUS_KM = 30


def distance_km(lat1, lng1, lat2, lng2):
    """Great-circle distance between two points (haversine formula)."""
    lat1, lng1, lat2, lng2 = map(math.radians, [lat1, lng1, lat2, lng2])
    a = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lng2 - lng1) / 2) ** 2
    return 6371 * 2 * math.asin(math.sqrt(a))


stations["km_to_heilbronn"] = stations.apply(
    lambda s: distance_km(*HEILBRONN, s["lat"], s["lng"]) if pd.notna(s["lat"]) else None, axis=1
)
heilbronn_stations = stations[stations["km_to_heilbronn"] <= RADIUS_KM].sort_values("km_to_heilbronn")
heilbronn_stations[["station_id", "name", "km_to_heilbronn", "last_movement", "last_temperature"]]

In [ ]:
hn_map = folium.Map(location=HEILBRONN, zoom_start=10, tiles=None)
for _, s in heilbronn_stations.iterrows():
    folium.Marker([s["lat"], s["lng"]], popup=f"<b>{s['name']}</b><br>{s['station_id']}").add_to(hn_map)
hn_map

## 3. A closer look at one station

We'll use the most recently active Heilbronn station. You can also paste any `station_id` from the tables above.

In [ ]:
station_id = heilbronn_stations.sort_values("last_movement", ascending=False).iloc[0]["station_id"]
# station_id = "699b7f2c-3855-4273-af7f-ed61a54541f9"  # or pick one by hand

station = get(f"/station/{station_id}", movements=20)

print(station["name"], "→", station["location"])
print("Paging info:", station["movementsMeta"])

### 3a. Movements (bird visits)

`measurements.movements` holds the newest movements, newest first. Each one can have **several detections**,
because the AI lists every species it thinks it might be, each with a confidence `score`. In this table we keep only
the **best guess** for each movement.

In [ ]:
def movements_to_df(movements):
    """One row per movement, with the AI's top guess as the species."""
    rows = []
    for m in movements:
        best = max(m.get("detections") or [], key=lambda d: d["score"], default=None)
        rows.append(
            {
                "mov_id": m["mov_id"],
                "station_id": m["station_id"],
                "start_date": m["start_date"],
                "weight_g": m.get("weight"),
                "species_latin": best["latinName"] if best else None,
                "species_german": best["germanName"] if best else None,
                "score": best["score"] if best else None,
                "n_detections": len(m.get("detections") or []),
                "validated": "validation" in m,
                "temperature": (m.get("environment") or {}).get("temperature"),
                "video": m.get("video"),
            }
        )
    df = pd.DataFrame(rows)
    if not df.empty:
        df["start_date"] = pd.to_datetime(df["start_date"], format="ISO8601")
    return df


recent = movements_to_df(station["measurements"]["movements"])
recent

Every movement links to its video, so you can watch it right here:

In [ ]:
from IPython.display import Video

first_with_video = recent.dropna(subset=["video"]).iloc[0]
print(first_with_video["start_date"], "-", first_with_video["species_german"] or first_with_video["species_latin"])
Video(first_with_video["video"], embed=False, width=480)

### 3b. Paging through older movements

`?movements=N` limits how many movements come back, and `&movementsOffset=K` skips the first K.
Movements are sorted **newest first**, so to get everything since a given date we keep fetching pages
until we reach a movement that's older than that date. This is a lot lighter than downloading the full
history with `/movement/{id}`.

In [ ]:
def fetch_movements_since(station_id, since, page_size=500):
    """Return all raw movements of a station with start_date >= since (a datetime)."""
    result, offset = [], 0
    while True:
        page = get(f"/station/{station_id}", movements=page_size, movementsOffset=offset)
        movements = page.get("measurements", {}).get("movements", [])
        for m in movements:
            if pd.to_datetime(m["start_date"], format="ISO8601") < since:
                return result  # everything after this is older → done
            result.append(m)
        if not page.get("movementsMeta", {}).get("hasMore"):
            return result
        offset += page_size


last_week = movements_to_df(fetch_movements_since(station_id, datetime.now() - timedelta(days=7)))
print(len(last_week), "movements in the last 7 days")
last_week["species_latin"].value_counts().head(10)

### 3c. Daily species counts (`count`)

The station response also includes `count`, a pre-computed daily summary: `{ "YYYY-MM-DD": {"sum": …, "birds": [{latinName, germanName, amount}]}}`.
It's the quickest way to get long time series without fetching any movements.

In [ ]:
daily = pd.DataFrame(
    [
        {"date": day, "latinName": b["latinName"], "germanName": b["germanName"], "amount": b["amount"]}
        for day, info in station.get("count", {}).items()
        for b in info["birds"]
    ]
)
daily["date"] = pd.to_datetime(daily["date"])

top_species = daily.groupby("latinName")["amount"].sum().nlargest(5).index
(
    daily[daily["latinName"].isin(top_species)]
    .pivot_table(index="date", columns="latinName", values="amount", aggfunc="sum")
    .resample("W").sum()
    .plot(figsize=(12, 4), title=f"Weekly visits of the 5 most common species – {station['name']}")
);

### 3d. Statistics and sensor data

`/statistics/{id}` gives a summary of the station. `/environment/{id}` returns **every** temperature/humidity reading,
which can be tens of MB, so download it once and reuse it.

In [ ]:
stats = get(f"/statistics/{station_id}")
{k: stats[k] for k in ["name", "numberOfMovements", "perDay", "maxTemp", "minTemp", "maxHum", "minHum"]}

In [ ]:
environment = pd.DataFrame(get(f"/environment/{station_id}"))
environment["date"] = pd.to_datetime(environment["date"], format="ISO8601")
environment = environment.set_index("date").sort_index()

environment[["temperature", "humidity"]].resample("D").mean().plot(
    subplots=True, figsize=(12, 5), title=f"Daily mean temperature & humidity – {station['name']}"
);

## 4. Combining endpoints: great tit (*Parus major*) sightings from the last 30 days

**Question:** *Where and when was a great tit (Kohlmeise) seen in the last 30 days, at any station?*

No single endpoint answers this, so we combine them:

1. `GET /station` → all stations with their **location** and **last movement**.
   Stations whose last movement is older than 30 days can't have any recent sightings, so we skip them.
   That saves a lot of requests.
2. `GET /station/{id}?movements=…&movementsOffset=…` → page through each remaining station's movements until we reach the 30-day cutoff.
3. Keep the movements where *Parus major* is the AI's **top guess** (and above a minimum confidence), then add the station's name and location.

> **What counts as a sighting?** The detections are AI guesses. Here a movement counts if *Parus major* is the most likely
> species and the score is at least `MIN_SCORE`. You can raise `MIN_SCORE` for fewer but more reliable sightings, or set
> `TOP_GUESS_ONLY = False` to count any detection of the species. Movements with `validated = True` were checked by a person.

In [ ]:
SPECIES = "Parus major"   # try "Cyanistes caeruleus" (Blaumeise), "Passer domesticus" (Haussperling), …
DAYS = 30
MIN_SCORE = 0.3
TOP_GUESS_ONLY = True
ONLY_HEILBRONN = False   # True → only search the stations from section 2

since = datetime.now() - timedelta(days=DAYS)

# Step 1: candidate stations (active within the time window)
candidates = heilbronn_stations if ONLY_HEILBRONN else stations
candidates = candidates[candidates["last_movement"] >= since]
print(f"Searching {len(candidates)} stations active since {since:%Y-%m-%d} …")

# Step 2 + 3: page through movements and keep the matching ones
sightings = []
for _, s in candidates.iterrows():
    for m in fetch_movements_since(s["station_id"], since):
        detections = sorted(m.get("detections") or [], key=lambda d: d["score"], reverse=True)
        if TOP_GUESS_ONLY:
            detections = detections[:1]
        match = next((d for d in detections if d["latinName"] == SPECIES and d["score"] >= MIN_SCORE), None)
        if match:
            sightings.append(
                {
                    "date": m["start_date"],
                    "station_name": s["name"],
                    "station_id": s["station_id"],
                    "lat": s["lat"],
                    "lng": s["lng"],
                    "score": round(match["score"], 2),
                    "weight_g": round(m["weight"], 1) if m.get("weight") is not None else None,
                    "validated": "validation" in m,
                    "video": m.get("video"),
                }
            )

sightings = pd.DataFrame(sightings)
if sightings.empty:
    print("No sightings found. Try lowering MIN_SCORE or increasing DAYS.")
else:
    sightings["date"] = pd.to_datetime(sightings["date"], format="ISO8601")
    sightings = sightings.sort_values("date", ascending=False).reset_index(drop=True)
    print(f"{len(sightings)} sightings of {SPECIES} at {sightings['station_id'].nunique()} stations")
sightings.head(20)

Sightings per station:

In [ ]:
sightings.groupby(["station_name", "lat", "lng"]).agg(
    sightings=("date", "size"), first=("date", "min"), last=("date", "max")
).sort_values("sightings", ascending=False)

…and on a map (circle size = number of sightings):

In [ ]:
per_station = sightings.groupby(["station_name", "lat", "lng"]).size().reset_index(name="n")

sightings_map = folium.Map(location=[per_station["lat"].mean(), per_station["lng"].mean()], zoom_start=6, tiles=None)
for _, row in per_station.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lng"]],
        radius=4 + 3 * math.log1p(row["n"]),
        color="orange",
        fill=True,
        popup=f"<b>{row['station_name']}</b><br>{row['n']} × {SPECIES}",
    ).add_to(sightings_map)
sightings_map

At what time of day do they show up?

In [ ]:
sightings["date"].dt.hour.value_counts().sort_index().plot.bar(
    figsize=(10, 3), xlabel="hour of day (station local time)", ylabel="sightings", title=SPECIES
);

### Save your results

Save the results to a CSV so you don't have to download them again. In Colab you can download the file from the folder icon on the left,
or with `files.download(...)`.

In [ ]:
filename = f"sightings_{SPECIES.replace(' ', '_')}_last_{DAYS}_days.csv"
sightings.to_csv(filename, index=False)
stations.to_csv("birdiary_stations.csv", index=False)
print("Saved", filename, "and birdiary_stations.csv")

# Uncomment in Colab to download straight away:
# from google.colab import files
# files.download(filename)

## 5. Ideas to get you started

* 🌡️ **Weather & birds:** do more birds visit when it's cold? Combine `environment` with the movements of a station.
* ⚖️ **Weight:** plot the `weight` of a species over the year. Can you tell great tits from blue tits by weight alone?
* 🗺️ **Species maps:** run section 4 for several species and compare where they show up.
* 🕐 **Daily rhythms:** which species are early birds, and which come in the afternoon?
* 🤖 **AI vs. humans:** compare `detections` with `validation` for validated movements. How often is the AI right?
* 🏙️ **Heilbronn dashboard:** build a small web app. The API sends `Access-Control-Allow-Origin: *`, so you can call it directly from browser JavaScript.

### Good to know
* `start_date` comes from the station's own clock (usually German local time). `createdAt` is the upload time and comes in mixed formats, so use `start_date` instead.
* `germanName` is sometimes empty (for species the model knows but that are rare in Germany).
* Some stations are test or exhibition devices (`type == "exhibit"`, or names like *test*).